# US Flight ARRIVAL Delay Analysis (2015)

**Goal:** understand what drives arrival delays and test whether we can predict them.

**Sections**
1. Setup & data loading
2. Cleaning
3. Exploratory analysis (airlines, months, distribution, correlations)
4. Feature engineering
5. Hypothesis tests (t-test, ANOVA, correlation)


## 1. Setup & data loading

In [ ]:

import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

DATA_DIR = Path(
    os.environ.get(
        "FLIGHT_DATA_DIR",
        r"C:\Users\DELL\Desktop\ML PROJECT DATASET"
    )
)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
flights=pd.read_csv("C:\Users\DELL\Desktop\ML PROJECT DATASET\flights.csv",low_memory=False)
airlines=pd.read_csv("C:\Users\DELL\Desktop\ML PROJECT DATASET\airlines.csv")
airlines.rename(columns={"IATA_CODE":"AIRLINE","AIRLINE":"AiRLINE_NAME"},inplace=True)
df=pd.merge(flights,airlines,on="AIRLINE",how="left")
airports=pd.read_csv("C:\Users\DELL\Desktop\ML PROJECT DATASET\airports.csv")
airports.drop(columns="AIRPORT",inplace=True)
airports.rename(columns={"IATA_CODE":"ORIGIN_AIRPORT","CITY":"ORIGIN_CITY","STATE":"ORIGIN_STATE","COUNTRY":"ORIGIN_COUNTRY","LATITUDE":"ORIGIN_LATITUDE","LONGITUDE":"ORIGIN_LONGITUDE"},inplace=True)
data=pd.merge(df,airports,on="ORIGIN_AIRPORT",how="left")
data=data.sample(n=100_000,random_state=42).reset_index(drop=True)

In [ ]:
print("number of rows:",data.shape[0])
print("number of columns:",data.shape[1])
print("data's info:",data.info(verbose=True,show_counts=True))
data.head()

## 2. Cleaning

* Duplicates are dropped (the original `data.drop_duplicates()` call did not save its result, so nothing was actually removed).
* The five delay-cause columns are only filled by the airline when a flight arrives 15+ minutes late, so a missing value means "no delay of that type" → filled with 0.
* Cancelled / diverted flights have no arrival information, so they are removed by the `dropna` below. **This means the analysis covers completed flights only.**

In [ ]:
rows_before = len(data)

data = data.drop_duplicates()          
data.fillna({
    "AIR_SYSTEM_DELAY":0,
    "SECURITY_DELAY":0,
    "AIRLINE_DELAY":0,
    "LATE_AIRCRAFT_DELAY":0,
    "WEATHER_DELAY":0,
    "CANCELLATION_REASON":"NOT CANCELLED"
},inplace=True)
data.dropna(subset=["DEPARTURE_DELAY","ARRIVAL_DELAY","TAXI_OUT","TAXI_IN","AIR_TIME","ELAPSED_TIME","WHEELS_ON","WHEELS_OFF","ORIGIN_LATITUDE","ORIGIN_LONGITUDE","ORIGIN_CITY","ORIGIN_STATE","ORIGIN_COUNTRY"],inplace=True)
data = data.reset_index(drop=True)

print(f"rows before: {rows_before:,} | rows after: {len(data):,} | removed: {rows_before-len(data):,} ({(rows_before-len(data))/rows_before:.1%})")
print("remaining missing values:", int(data.isna().sum().sum()))
data.sample(10, random_state=RANDOM_STATE)

## 3. Exploratory analysis

### Which airline has the highest average arrival delay?

In [ ]:
y=data.groupby("AiRLINE_NAME")["ARRIVAL_DELAY"].mean().sort_values(ascending=True)
plt.figure(figsize=(9,6))
plt.barh(y.index, y.values)
plt.xlabel("Mean arrival delay (minutes)")
plt.title("Average arrival delay by airline")
plt.tight_layout()
plt.show()

print("Highest:", y.idxmax(), f"({y.max():.1f} min)")
print("Lowest :", y.idxmin(), f"({y.min():.1f} min)")

### Monthly arrival delay trend

In [ ]:
y=data.groupby("MONTH")["ARRIVAL_DELAY"].mean()
plt.figure(figsize=(9,4))
plt.bar(y.index, y.values)
plt.xticks(range(1,13))
plt.xlabel("Month")
plt.ylabel("Mean arrival delay (minutes)")
plt.title("Average arrival delay by month")
plt.show()
print("Worst month:", y.idxmax(), "| Best month:", y.idxmin())

### Distribution of arrival delay

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data["ARRIVAL_DELAY"].clip(-60, 300), bins=80, ax=ax[0])
ax[0].set_title("Arrival delay (clipped to -60..300 min)")
ax[0].set_xlabel("minutes")
sns.boxplot(x=data["ARRIVAL_DELAY"], showfliers=False, ax=ax[1])
ax[1].set_title("Arrival delay (outliers hidden)")
plt.tight_layout()
plt.show()

d = data["ARRIVAL_DELAY"]
print(f"mean={d.mean():.1f} | median={d.median():.1f} | skew={d.skew():.1f}")
print(f"flights arriving early (<0 min): {(d<0).mean():.1%}")
print(f"flights 15+ min late (official 'delayed'): {(d>=15).mean():.1%}")

### Skewness & transformation

Most delay columns are heavily right-skewed. The original code applied `log1p` to every skewed column, which
* produced `divide by zero` / `invalid value` warnings and **NaN values**, because `ARRIVAL_DELAY` and `DEPARTURE_DELAY` contain negative numbers (early flights) and `log1p` is undefined below -1, and
* silently dropped all those early flights from the correlation table.

Below, `log1p` is applied only to non-negative skewed columns, and the two delay targets are kept raw.

In [ ]:
num=data.select_dtypes(include="number").copy()
num.skew()

In [ ]:
num=num.drop(columns=["YEAR","DIVERTED","CANCELLED"],errors='ignore')
skewed=num.skew()
keep_raw=["ARRIVAL_DELAY","DEPARTURE_DELAY"]          
pos_skew=[c for c in skewed[skewed>0.5].index if c not in keep_raw and num[c].min()>=0]

num_t=num.copy()
for c in pos_skew:
    num_t[c]=np.log1p(num_t[c])

print("log1p applied to:", pos_skew)
print("NaNs introduced:", int(num_t.isna().sum().sum()))
pd.DataFrame({"skew_before":num[pos_skew].skew(),"skew_after":num_t[pos_skew].skew()}).round(2)

### Correlation with arrival delay

The original `sns.pairplot(num.corr())` plotted a correlation *matrix* as if it were data (a 6250x6250 px figure with no meaning), so it is replaced by a heatmap of each feature's correlation with `ARRIVAL_DELAY`.
(The original heatmap used `AIRLINE_DELAY` as the target, which did not match the project goal.)

In [ ]:
corr=pd.DataFrame({
    "pearson" : num.corr()["ARRIVAL_DELAY"],
    "spearman": num.corr(method="spearman")["ARRIVAL_DELAY"],
}).drop("ARRIVAL_DELAY").sort_values("pearson",ascending=False)

plt.figure(figsize=(6,9))
sns.heatmap(
    corr,
    annot=True,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)
plt.title("Correlation of Numerical Features with Arrival Delay")
plt.show()

## 4. Feature engineering

* **Season:** the original map skipped **November**, so every November flight became `NaN`. Fixed below.
* **Scheduled hour:** extracted from `SCHEDULED_DEPARTURE` (hhmm format) - late-day flights usually inherit earlier delays.
* The original `pd.get_dummies(data[...].unique())` cells encoded the *list of unique names* instead of the data, were never used, and the `season_onehot["False"]/["True"]` columns were hard-coded. They are removed; encoding is done properly inside the model pipeline (section 6).

In [ ]:
season_map={
    12:'Winter',1:"Winter",2:"Winter",
    3:'Spring',4:'Spring',5:'Spring',
    6:'Summer',7:"Summer",8:'Summer',
    9:'Fall',10:'Fall',11:'Fall'         
}
data["Season"]=data["MONTH"].map(season_map)
data["HOUR"]=(data["SCHEDULED_DEPARTURE"]//100)%24

assert data["Season"].notna().all(), "some months were not mapped"
print(data["Season"].value_counts())

## 5. Hypothesis tests

### Test 1 - Winter vs Summer (Welch t-test)

* **H0:** winter and summer flights have the same mean arrival delay
* **H1:** the means differ

In [ ]:
winter = data.loc[data["Season"] == "Winter", "ARRIVAL_DELAY"].dropna()
summer = data.loc[data["Season"] == "Summer", "ARRIVAL_DELAY"].dropna()

t_value, p_value = stats.ttest_ind(winter, summer, equal_var=False)
p_ttest = p_value            

print("Winter mean delay:", winter.mean())
print("Summer mean delay:", summer.mean())
print("T-statistic:", t_value)
print("P-value:", p_value)
if p_value<0.05:
    print("null hypothesis rejected")
else:
    print("fail to reject the null hypothesis (not 'accept' - absence of evidence is not proof of equality)")


pooled_sd = np.sqrt((winter.var()+summer.var())/2)
print(f"Cohen's d: {(winter.mean()-summer.mean())/pooled_sd:.3f}")
u, p_mw = stats.mannwhitneyu(winter, summer)
print(f"Mann-Whitney U p-value (no normality assumption): {p_mw:.4f}")

### Test 2 - Do airlines differ? (one-way ANOVA)

`f_oneway` was imported in the original notebook but never used. 
* **H0:** all airlines have the same mean arrival delay

In [ ]:
groups=[g.values for _,g in data.groupby("AiRLINE_NAME")["ARRIVAL_DELAY"]]
f_stat,p_anova=f_oneway(*groups)
h_stat,p_kw=stats.kruskal(*groups)          # robust to skew

grand=data["ARRIVAL_DELAY"].mean()
ss_between=sum(len(g)*(g.mean()-grand)**2 for g in groups)
ss_total=((data["ARRIVAL_DELAY"]-grand)**2).sum()

print(f"ANOVA F={f_stat:.2f}, p={p_anova:.3g}")
print(f"Kruskal-Wallis H={h_stat:.2f}, p={p_kw:.3g}")
print(f"Eta-squared (share of variance explained by airline alone): {ss_between/ss_total:.2%}")
print("Reject H0" if p_anova<0.05 else "Fail to reject H0")

### Test 3 - Which delay cause moves with arrival delay?

The original markdown claimed *"late-arriving aircraft has the highest positive correlation"*. That is only true **among the five cause columns**: `DEPARTURE_DELAY` correlates more strongly than any of them. The table below shows both.

* **H0:** correlation = 0

In [ ]:
x = data["LATE_AIRCRAFT_DELAY"]
y = data["ARRIVAL_DELAY"]

valid = data[["LATE_AIRCRAFT_DELAY", "ARRIVAL_DELAY"]].dropna()

r, p_value = stats.pearsonr(
    valid["LATE_AIRCRAFT_DELAY"],
    valid["ARRIVAL_DELAY"]
)

print("Correlation:", r)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject the null hypothesis")
    print("The correlation is statistically significant")
else:
    print("Fail to reject the null hypothesis")
    print("Insufficient evidence of a significant correlation")

rows=[]
for c in ["DEPARTURE_DELAY","AIR_SYSTEM_DELAY","SECURITY_DELAY","AIRLINE_DELAY","LATE_AIRCRAFT_DELAY","WEATHER_DELAY"]:
    rr,pp=stats.pearsonr(data[c],data["ARRIVAL_DELAY"])
    rows.append((c,rr,pp))
cause_tbl=pd.DataFrame(rows,columns=["feature","pearson_r","p_value"]).sort_values("pearson_r",ascending=False)
display(cause_tbl.round(4))

**Caveat:** the cause columns are recorded only for flights that arrive 15+ minutes late (zeros otherwise) and their minutes add up to the arrival delay, so their correlation with `ARRIVAL_DELAY` is partly mechanical. They describe *why* a flight was late, not something known beforehand, which is why they are excluded from the prediction model below.